In [ ]:
from moviepy.editor import ImageClip

/usr/local/lib/python3.13/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':

  from pkg_resources import resource_stream, resource_exists



In [ ]:
from PIL import Image
#Image.new("RGB", (480, 640), color="black").save("black.png")

In [12]:
create_video_with_slide("3ef08bb5-a298-11f1-9cae-6a4b6c56230e_photo.jpg", "output.mp4", duration=12, width=480, height=640, direction='left_to_right')

Moviepy - Building video output.mp4.
Moviepy - Writing video output.mp4



Moviepy - Done !
Moviepy - video ready output.mp4


In [ ]:
# кроп с анимацией
from moviepy.editor import ImageClip, CompositeVideoClip

def create_video_with_slide(image_path, output_path, duration=15, width=480, height=640, slide_percent=0.15, direction='left_to_right'):
    """
    direction: 'left_to_right' (выезжает слева) или 'right_to_left' (выезжает справа)
    slide_percent: доля движения (0.15 = 15%)
    """

    # 1. Загружаем картинку
    clip = ImageClip(image_path)

    # 2. Рассчитываем размер для анимации
    # Чтобы было красиво, мы можем немного увеличить картинку перед анимацией,
    # либо двигать оригинальную. Здесь двигаем оригинальную, но проверяем пропорции.

    aspect_in = clip.w / clip.h
    aspect_out = width / height

    # Логика масштабирования (чтобы картинка была не меньше кадра для плавного выезда)
    if aspect_in > aspect_out:
        new_h = height
        new_w = int(clip.w * (new_h / clip.h))
    else:
        new_w = width
        new_h = int(clip.h * (new_w / clip.w))

    resized_clip = clip.resize(width=new_w, height=new_h)

    # 3. Рассчитываем расстояние движения (15% от ширины кадра)
    move_distance = int(width * slide_percent)

    # Функция позиции в зависимости от времени (t в секундах)
    def get_position(t, total_duration, start_x, end_x):
        # Линейная интерполяция: от start_x до end_x за total_duration секунд
        progress = min(t / total_duration, 1.0)
        return (start_x + (end_x - start_x) * progress, 'center') # Y остается по центру

    # Определяем начальную и конечную позицию по X
    if direction == 'left_to_right':
        # Начинаем ЗА левым краем (-move_distance), заканчиваем на центре (0) или чуть дальше
        start_x = -move_distance
        end_x = 0
    elif direction == 'right_to_left':
        # Начинаем ЗА правым краем, заканчиваем на центре
        start_x = width + move_distance
        end_x = 0
    else:
        raise ValueError("direction должен быть 'left_to_right' или 'right_to_left'")

    # Применяем анимацию позиции
    animated_clip = resized_clip.set_position(lambda t: get_position(t, duration, start_x, end_x))

    # Важно: устанавливаем длительность и обрезаем клип ровно по времени
    final_clip = animated_clip.set_duration(duration)

    # Рендер
    final_clip.write_videofile(output_path, fps=24, codec='libx264', audio_codec='aac')

    # Закрытие ресурсов
    final_clip.close()
    animated_clip.close()
    resized_clip.close()
    clip.close()

# Пример использования:
# create_video_with_slide("input.jpg", "output.mp4", duration=12, direction='left_to_right')
